# Prune a dataset by matching `input` values

1. Read the `input` field from the **origin** dataset for a given list of row numbers.
2. For each of those `input` values, drop every row in the **dataset to clean** whose own `input` field matches.
3. Save the cleaned dataset next to the input with `_pruned` appended to the stem.

Row numbers are **0-based positional** indices into the origin data rows (the CSV header is not counted).

In [2]:
from pathlib import Path
import csv
import sys

# Raise the default 128 KB CSV field limit (long generated text can exceed it).
_limit = sys.maxsize
while True:
    try:
        csv.field_size_limit(_limit)
        break
    except OverflowError:
        _limit //= 2

## Configuration

In [6]:
# Origin dataset: row numbers below are read from here.
ORIGIN_PATH = Path("/Users/bastianellie/Library/CloudStorage/OneDrive-SharedLibraries-ReedElsevierGroupICOReedElsevierInc/OG-Data Science Research Content - Researcher AI/evaluation/writing-coach/q1-beta-release/benchmark/writing-coach-query-inputtext-natural+synth.csv")

# Dataset to clean: rows whose `input` matches are removed from here.
CLEAN_PATH = Path("/Users/bastianellie/Library/CloudStorage/OneDrive-SharedLibraries-ReedElsevierGroupICOReedElsevierInc/OG-Data Science Research Content - Researcher AI/evaluation/writing-coach/q2-full-release/benchmark/positional_query_0615.csv")

# 0-based positional row numbers in the origin dataset.
ROW_NUMBERS = [
    93, 94, 95, 151, 152, 154, 155, 156, 199, 200, 201, 202, 203, 207, 208, 215, 216, 217, 228, 229, 237, 241, 245
]

# Name of the column holding the text to match on.
INPUT_FIELD = "input"

## Step 1 — collect the target `input` values from the origin dataset

In [7]:
def collect_inputs(path: Path, row_numbers: set[int], field: str) -> set[str]:
    """Return the ``field`` values found at the given 0-based data-row indices."""
    targets = {}
    with open(path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        if field not in (reader.fieldnames or []):
            raise KeyError(f"Column {field!r} not in {path.name}: {reader.fieldnames}")
        for index, row in enumerate(reader):
            if index in row_numbers:
                targets[index] = row[field]

    missing = row_numbers - targets.keys()
    if missing:
        print(f"  warning: row numbers out of range (ignored): {sorted(missing)}")
    return set(targets.values())


row_set = set(ROW_NUMBERS)
target_inputs = collect_inputs(ORIGIN_PATH, row_set, INPUT_FIELD)
print(f"Collected {len(target_inputs)} distinct {INPUT_FIELD!r} value(s) from {len(row_set)} row(s).")

Collected 23 distinct 'input' value(s) from 23 row(s).


## Step 2 — remove matching rows from the dataset to clean and save

In [8]:
def pruned_path(path: Path) -> Path:
    """Return ``path`` with ``_pruned`` appended to the stem, in the same directory."""
    return path.with_name(f"{path.stem}_pruned{path.suffix}")


def prune_by_input(path: Path, targets: set[str], field: str) -> Path:
    """Drop rows of ``path`` whose ``field`` value is in ``targets``; write ``_pruned`` copy."""
    output = pruned_path(path)
    kept = deleted = 0
    matched = set()

    with open(path, newline="", encoding="utf-8") as in_f, \
            open(output, "w", newline="", encoding="utf-8") as out_f:
        reader = csv.DictReader(in_f)
        if field not in (reader.fieldnames or []):
            raise KeyError(f"Column {field!r} not in {path.name}: {reader.fieldnames}")
        writer = csv.DictWriter(out_f, fieldnames=reader.fieldnames)
        writer.writeheader()
        for row in reader:
            value = row[field]
            if value in targets:
                matched.add(value)
                deleted += 1
            else:
                writer.writerow(row)
                kept += 1

    not_found = targets - matched
    if not_found:
        print(f"{len(not_found)} {field!r} value(s) not found in {path.name}:")
        for value in not_found:
            preview = value if len(value) <= 200 else value[:200] + "..."
            print(f"  - {preview!r}")
    else:
        print(f"All {len(targets)} target {field!r} value(s) were found.")

    print(f"Kept {kept}, deleted {deleted} -> {output}")
    return output


prune_by_input(CLEAN_PATH, target_inputs, INPUT_FIELD)

21 'input' value(s) not found in positional_query_0615.csv:
  - 'During the building’s thirty-year lifespan, maintenance reports have recorded multiple incidents of roof leakage, faulty insulation, and inconsistent heating performance. The facility houses approxima...'
  - 'This book consists of magazine and newspaper articles, interviews, record reviews, excerpts from books, and other material related to the history of The Doors, to whose memory it is dedicated.This boo...'
  - 'Right now, the lab is kind of messy. There’s equipment all over the place, and some samples haven’t been labeled right, which makes it hard to figure out which ones belong to which experiments. The te...'
  - "The Lakers experienced a tremendous run of success in the 1980s, winning five NBA Championships and making the NBA Finals every year but two (1981 and 1986). This level of success raised The Forum's p..."
  - 'While I can see how the reluctance to speak out may come from a place of good intentions, in no

PosixPath('/Users/bastianellie/Library/CloudStorage/OneDrive-SharedLibraries-ReedElsevierGroupICOReedElsevierInc/OG-Data Science Research Content - Researcher AI/evaluation/writing-coach/q2-full-release/benchmark/positional_query_0615_pruned.csv')